# Libraries

In [132]:
import requests
from bs4 import BeautifulSoup
from urllib.parse import urljoin
import pdfplumber
import tempfile
from dataclasses import dataclass
from datetime import datetime
from enum import Enum
import pandas as pd
from IPython.display import display

# Constants
SCHEDULES_URL = "http://www.caecis.ugto.mx/caecis/pages/horarios.aspx"

# Extract PDFs only from DICIS

In [133]:
@dataclass
class AnchorInfo:
  name: str
  href: str

In [134]:
response = requests.get(SCHEDULES_URL)
html = response.text

soup = BeautifulSoup(html, "html.parser")

target_element = soup.find("font", string=lambda text: "Sede Salamanca Enero" in text if text else False)

# Get closest table
anchors_info: list[AnchorInfo] = []
table = target_element.find_next("table")

# Get all anchors in a table
children_anchors = table.find_all("a")

for a in children_anchors:
  if ".pdf" in a.get("href"):
    anchors_info.append(AnchorInfo(name=a.text.strip(), href=urljoin(SCHEDULES_URL, a.get("href"))))

print(anchors_info)

[AnchorInfo(name='Ingeniería Mecánica', href='http://www.caecis.ugto.mx/caecis/Documentos/2026/Horarios/EJ26/DI/LIME_ACT_2.pdf'), AnchorInfo(name='Ingeniería Eléctrica', href='http://www.caecis.ugto.mx/caecis/Documentos/2026/Horarios/EJ26/DI/LIE_ACT_2.pdf'), AnchorInfo(name='Ingeniería en Comunicaciones y Electrónica', href='http://www.caecis.ugto.mx/caecis/Documentos/2026/Horarios/EJ26/DI/LICE_SAL.pdf'), AnchorInfo(name='Ingeniería en Mecatrónica', href='http://www.caecis.ugto.mx/caecis/Documentos/2026/Horarios/EJ26/DI/LIMT.pdf'), AnchorInfo(name='Ingeniería en Sistemas Computacionales', href='http://www.caecis.ugto.mx/caecis/Documentos/2026/Horarios/EJ26/DI/LISC_SAL.pdf'), AnchorInfo(name='Licenciatura en Gestión Empresarial', href='http://www.caecis.ugto.mx/caecis/Documentos/2026/Horarios/EJ26/DI/LGE_SAL_ACT_2.pdf'), AnchorInfo(name='Licenciatura en Artes Digitales', href='http://www.caecis.ugto.mx/caecis/Documentos/2026/Horarios/EJ26/DI/LAD.pdf'), AnchorInfo(name='Ingeniería en Dat

# Download PDFs and parse them

In [135]:
@dataclass
class TimeRange:
    from_: datetime
    to: datetime

class Day(str, Enum):
  MONDAY = "MONDAY"
  TUESDAY = "TUESDAY"
  WEDNESDAY = "WEDNESDAY"
  THURSDAY = "THURSDAY"
  FRIDAY = "FRIDAY"
  SATURDAY = "SATURDAY"

@dataclass
class Professor:
  names: str
  last_names: str
  honorific: str

@dataclass 
class Schedule:
  day: Day
  timeRange: TimeRange

@dataclass
class Class:
  uda: str
  classroom: str
  schedules: list[Schedule]
  professor: Professor

@dataclass 
class Course:
  name: str
  classes: list[Class]
  updatedAt: datetime | None

In [136]:
MONTHS = {
  "enero": 1, "febrero": 2, "marzo": 3,
  "abril": 4, "mayo": 5, "junio": 6,
  "julio": 7, "agosto": 8, "septiembre": 9,
  "octubre": 10, "noviembre": 11, "diciembre": 12
}

def parse_spanish_date(s):
  parts = s.lower().replace("de", "").split()
  day = int(parts[0])
  month = MONTHS[parts[1]]
  year = int(parts[2])
  return datetime(year, month, day)

def clean(text):
  return " ".join(text.split())

def extract_days_idx(headers):
  days_idx = {}

  for i, h in enumerate(headers):
    if h and "LUN" in h:
      days_idx["MONDAY"] = (i, i+1)
    elif h and "MAR" in h:
      days_idx["TUESDAY"] = (i, i+1)
    elif h and "MIÉ" in h:
      days_idx["WEDNESDAY"] = (i, i+1)
    elif h and "JUE" in h:
      days_idx["THURSDAY"] = (i, i+1)
    elif h and "VIE" in h:
      days_idx["FRIDAY"] = (i, i+1)
    elif h and "SÁB" in h:
      days_idx["SATURDAY"] = (i, i+1)
  return days_idx

def format_professor_name(name) -> Professor:
  name = clean(name)

  # The honorific is after the "," character
  parts = name.split(",")

  # The two first words are the last names
  name_parts = parts[0].split()
  last_names = " ".join(name_parts[:2]).title()
  names = " ".join(name_parts[2:]).title()

  honorific = parts[1].strip() if len(parts) > 1 else ""

  return Professor(names=names, last_names=last_names, honorific=honorific)

def upper_table(table):
  return [
    [
      cell.upper() if isinstance(cell, str) else cell
      for cell in row
    ]
    for row in table
  ]

In [137]:
def parse_table(_table) -> list[Class]:
  table = upper_table(_table)

  headers = table[0]
  rows = table[1:]

  classes: list[Class] = []
  
  uda_idx = headers.index("UDA")
  classroom_idx = headers.index("AULA")
  professor_idx = headers.index("PROFESOR")
  days_idx = extract_days_idx(headers)

  for row in rows:
    days = {}

    for day, (from_idx, to_idx) in days_idx.items():
      start = row[from_idx]
      end = row[to_idx]
      uda = row[uda_idx]
      classroom = row[classroom_idx]
      professor = row[professor_idx]

      if start and end and classroom and professor and uda:
        days[day] = TimeRange(from_=datetime.strptime(start, "%H:%M"), to=datetime.strptime(end, "%H:%M"))

        _class = Class(
          uda=clean(uda),
          classroom=clean(classroom),
          schedules=[Schedule(day=day, timeRange=days[day])],
          professor=format_professor_name(professor)
        )

        classes.append(_class)
  
  return classes

def download_and_parse_pdf(info: AnchorInfo) -> Course:
  with requests.get(info.href, stream=True) as r:
    r.raise_for_status()

    with tempfile.NamedTemporaryFile(suffix=".pdf") as tmp:
      for chunk in r.iter_content(8192):
        tmp.write(chunk)

      tmp.flush()

      return parse_pdf(tmp.name, info)

def get_updated_at_from_pdf(path: str) -> datetime | None:
  with pdfplumber.open(path) as pdf:
    for page in pdf.pages:
      # If is the first page (contains the course name)
      if page.page_number == 1:
        text = page.extract_text()

        # Find `Fecha de actualización:`
        updated_at_str = text.split("Fecha de actualización:")

        if len(updated_at_str) < 2:
          return None

        return parse_spanish_date(updated_at_str[1].split("\n")[0].strip())

def parse_pdf(path: str, info: AnchorInfo) -> Course:
  with pdfplumber.open(path) as pdf:
    updated_at = get_updated_at_from_pdf(path)
    classes: list[Class] = []

    for page in pdf.pages:
      tables = page.extract_tables()
      page_classes = parse_table(tables[0])
      classes.extend(page_classes)

    return Course(name=info.name, classes=classes, updatedAt=updated_at)

courses: list[Course] = []

for data in anchors_info:
  print(f"Processing {data.name}...")
  course = download_and_parse_pdf(data)
  courses.append(course)

print(courses)

Processing Ingeniería Mecánica...
Processing Ingeniería Eléctrica...
Processing Ingeniería en Comunicaciones y Electrónica...
Processing Ingeniería en Mecatrónica...
Processing Ingeniería en Sistemas Computacionales...
Processing Licenciatura en Gestión Empresarial...
Processing Licenciatura en Artes Digitales...
Processing Ingeniería en Datos e Ingeniería Artifícial...
[Course(name='Ingeniería Mecánica', classes=[Class(uda='ACTIVIDAD FÍSICA Y SALUD', classroom='CANCHAS', schedules=[Schedule(day='MONDAY', timeRange=TimeRange(from_=datetime.datetime(1900, 1, 1, 10, 0), to=datetime.datetime(1900, 1, 1, 12, 0)))], professor=Professor(names='Cristopher', last_names='Pérez Flores', honorific='LIC.')), Class(uda='ACTIVIDAD FÍSICA Y SALUD', classroom='CANCHAS', schedules=[Schedule(day='WEDNESDAY', timeRange=TimeRange(from_=datetime.datetime(1900, 1, 1, 8, 0), to=datetime.datetime(1900, 1, 1, 10, 0)))], professor=Professor(names='Cristopher', last_names='Pérez Flores', honorific='LIC.')), Clas

In [139]:
def format_day(day):
  if hasattr(day, "value"):
    return day.value.capitalize()
  return str(day).capitalize()

def display_courses(courses: list[Course]):
  rows = []

  for course in courses:
    for cls in course.classes:
      for sched in cls.schedules:
        rows.append({
          "Course": course.name,
          "UDA": cls.uda,
          "Classroom": cls.classroom,
          "Day": format_day(sched.day),
          "Start": sched.timeRange.from_.strftime("%H:%M"),
          "End": sched.timeRange.to.strftime("%H:%M"),
          "Professor": f"{cls.professor.honorific} {cls.professor.names} {cls.professor.last_names}",
          "Updated": course.updatedAt.strftime("%Y-%m-%d") if course.updatedAt else "-"
        })

  df = pd.DataFrame(rows)

  if df.empty:
    print("No courses")
    return

  # Orden opcional
  df = df.sort_values(["Course", "Day", "Start"])

  # Estilo bonito
  styled = (
    df.style
    .set_properties(**{"text-align": "left"})
    .set_table_styles([
      {"selector": "th", "props": [("text-align", "left"), ("font-weight", "bold")]},
      {"selector": "td", "props": [("padding", "6px")]}
    ])
  )

  display(styled)

display_courses(courses)

,Course,UDA,Classroom,Day,Start,End,Professor,Updated
178,Ingeniería Eléctrica,CÁLCULO INTEGRAL,A312,Friday,08:00,10:00,DR. José Merced Lozano García,2025-01-08
180,Ingeniería Eléctrica,CÁLCULO INTEGRAL,A206,Friday,08:00,10:00,M.I. Irving Willihado Reyes Buenfil,2025-01-08
208,Ingeniería Eléctrica,DISEÑO DE MÁQUINAS ELÉCTRICAS,LAB. IE2,Friday,08:00,10:00,DR. Iván Abel Hernández Robles,2025-01-08
259,Ingeniería Eléctrica,QUÍMICA UNIVERSITARIA,A207,Friday,08:00,10:00,ING. Maricela Vázquez Jaime,2025-01-08
265,Ingeniería Eléctrica,SEÑALES Y SISTEMAS,LAB. IE1,Friday,08:00,10:00,DR. Rafael Guzmán Cabrera,2025-01-08
185,Ingeniería Eléctrica,CÁLCULO Y MODELADO DE REDES ELÉCTRICAS,LAB. IE1,Friday,10:00,12:00,DR. Osvaldo Rodríguez Villalón,2025-01-08
191,Ingeniería Eléctrica,CIENCIA DE MATERIALES PARA INGENIERÍA,A312,Friday,10:00,12:00,M.C. Victoria Paola Cabrera Madera,2025-01-08
193,Ingeniería Eléctrica,CIENCIA DE MATERIALES PARA INGENIERÍA,A206,Friday,10:00,12:00,DRA. Amanda Enriqueta Violante Gavira,2025-01-08
218,Ingeniería Eléctrica,ELECTRÓNICA DE POTENCIA,LAB. IE2,Friday,10:00,12:00,DR. José Merced Lozano García,2025-01-08
255,Ingeniería Eléctrica,PROGRAMACIÓN EN INGENIERÍA ELÉCTRICA,A207,Friday,10:00,11:30,DR. Rafael Guzmán Cabrera,2025-01-08
